In [0]:
import dlt
from pyspark.sql.functions import col, upper, when, round as spark_round, sum as spark_sum, avg as spark_avg, count as spark_count

# ── BRONZE LAYER ──────────────────────────────────────────
@dlt.table(
    name="bronze_finance",
    comment="Raw finance transactions — ingested as-is"
)
def bronze_finance():
    data = [
        (1,  "Alice",   "credit", 1500.00, "2026-01-01", "groceries"),
        (2,  "Bob",     "debit",  200.00,  "2026-01-02", "utilities"),
        (3,  "Alice",   "credit", 3000.00, "2026-01-03", "salary"),
        (4,  "Charlie", "debit",  450.00,  "2026-01-04", "rent"),
        (5,  "Bob",     "credit", 2500.00, "2026-01-05", "salary"),
        (6,  "Alice",   "debit",  None,    "2026-01-06", "shopping"),
        (7,  "Charlie", "credit", 5000.00, "2026-01-07", "salary"),
        (8,  "Bob",     "debit",  150.00,  "2026-01-08", "groceries"),
        (9,  "Alice",   "credit", 1200.00, "2026-01-09", "freelance"),
        (10, "Charlie", "debit",  300.00,  "2026-01-10", "utilities"),
    ]
    return spark.createDataFrame(data,
        ["id", "customer", "type", "amount", "date", "category"])


# ── SILVER LAYER ──────────────────────────────────────────
# ── SILVER LAYER ──────────────────────────────────────────
@dlt.table(
    name="silver_finance",
    comment="Cleaned and validated transactions"
)
@dlt.expect("valid_date", "date IS NOT NULL")
@dlt.expect_or_drop("valid_amount", "amount IS NOT NULL")
@dlt.expect_or_fail("valid_type", "type IN ('credit', 'debit')")
def silver_finance():
    return (dlt.read("bronze_finance")
        .withColumn("customer", upper(col("customer")))
        .withColumn("transaction_flag",
            when(col("amount") >= 2000, "HIGH")
            .when(col("amount") >= 500,  "MEDIUM")
            .otherwise("LOW"))
    )


# ── GOLD LAYER ────────────────────────────────────────────
@dlt.table(
    name="gold_finance",
    comment="Business metrics per customer"
)
def gold_finance():
    return (dlt.read("silver_finance")
        .groupBy("customer")
        .agg(
            spark_round(spark_sum("amount"), 2).alias("total_amount"),
            spark_round(spark_avg("amount"), 2).alias("avg_amount"),
            spark_count("id").alias("total_transactions")
        )
        .orderBy("total_amount", ascending=False)
    )